# UrgeNurse — Análisis de rendimiento de modelos GGUF

Este notebook usa `llm_performance.py` como **paquete de funciones**
(`from . import llm_performance as lp`) para:

1. **cargar** los modelos GGUF (uno a uno, en CPU, ≤5 GB RAM),
2. **correr las pruebas** de triage / NER / SBAR con `lp.run_benchmark()`,
   que devuelve un `pandas.DataFrame` (sin escribir archivos),
3. **analizar las métricas**: elegir el mejor modelo con un score compuesto y
   generar imágenes comparativas de rendimiento, memoria y CPU.

> Requiere `llama-cpp-python`, `pandas` y los modelos GGUF en `../models`.
> El benchmark corre 100% en CPU y puede tardar varios minutos.

In [ ]:
# 1) Configuración: carpeta de modelos en ../models (relativo al notebook)
import os
from pathlib import Path

MODELS_DIR = (Path.cwd() / ".." / "models").resolve()
os.environ["LLM_PERF_MODELS_DIR"] = str(MODELS_DIR)
# "1" = descarga todos los modelos del catálogo que falten; "0" = solo los locales.
os.environ["LLM_PERF_DOWNLOAD"] = "1"

print("LLM_PERF_MODELS_DIR =", os.environ["LLM_PERF_MODELS_DIR"])
print("LLM_PERF_DOWNLOAD   =", os.environ["LLM_PERF_DOWNLOAD"])

In [ ]:
# 2) Importar el paquete de funciones del benchmark como `lp`
import sys
from pathlib import Path

# 'scripts' debe ser importable como paquete (tiene __init__.py): añadimos su
# carpeta padre (code/) al path y fijamos el paquete actual del notebook.
PKG_DIR = Path.cwd() if Path.cwd().name == "scripts" else Path.cwd() / "scripts"
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))
__package__ = PKG_DIR.name  # "scripts"

from . import llm_performance as lp

OP_NAMES = [op.op_name for op in lp.OPERATIONS]
print(f"{len(OP_NAMES)} operaciones registradas:")
print(OP_NAMES)

In [ ]:
# 2b) Información del computador (OS, CPU, RAM tipo/velocidad/disp., swap, GPU)
sysinfo = lp.print_system_info()
sysinfo

## Cargar modelos + correr pruebas → DataFrame

`lp.run_benchmark()` carga cada modelo del catálogo (en CPU, ≤5 GB RAM), ejecuta
las pruebas de triage / NER / SBAR, imprime el progreso y **devuelve un
`pandas.DataFrame`** (sin escribir ningún CSV). Ese `df` es el que analizan las
celdas siguientes.

> Es pesado: carga cada modelo en CPU y corre todas las operaciones.

In [ ]:
# 3) Cargar modelos + correr las pruebas → DataFrame (Jupyter permite `await` arriba)
df = await lp.run_benchmark()
df

In [ ]:
# 3) Librerías de análisis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
# 4) El DataFrame `df` viene directo de lp.run_benchmark() (sin CSV intermedio)
assert "df" in globals(), "Ejecuta antes la celda de lp.run_benchmark()"
print(f"DataFrame con {len(df)} modelos y {len(df.columns)} columnas")
df.head()

## Preparación y métricas agregadas

Descartamos los modelos que no cargaron (omitidos por presupuesto de RAM o errores)
y calculamos, por modelo, la **accuracy media** y la **latencia media por operación**.

In [ ]:
# 5) Limpieza y métricas derivadas (una fila por modelo × contexto)
df["loaded"] = df["load_time_ms"] > 0
ok = df[df["loaded"]].copy()
# Etiqueta única por (modelo, contexto): "<modelo> @<n_ctx>"
ok["model_short"] = (
    ok["model"].str.replace(".gguf", "", regex=False) + " @" + ok["n_ctx"].astype(str)
)

acc_cols = [f"{n}_acc" for n in OP_NAMES if f"{n}_acc" in ok.columns]
ms_cols = [f"{n}_ms" for n in OP_NAMES if f"{n}_ms" in ok.columns]

ok[acc_cols] = ok[acc_cols].apply(pd.to_numeric, errors="coerce")
ok[ms_cols] = ok[ms_cols].apply(pd.to_numeric, errors="coerce")

ok["mean_acc"] = ok[acc_cols].mean(axis=1)
ok["mean_op_ms"] = ok[ms_cols].mean(axis=1)

print(f"{len(ok)} ejecuciones cargadas de {len(df)} (modelo × contexto)")
if (~df["loaded"]).any():
    print("Omitidos / con error:")
    print(
        df.loc[~df["loaded"], ["model", "n_ctx", "load_error"]].to_string(index=False)
    )

ok[
    [
        "model_short",
        "n_ctx",
        "file_size_mb",
        "load_time_ms",
        "mem_after_load_mb",
        "mean_op_ms",
        "mean_acc",
    ]
]

In [ ]:
# 6) Desglose de accuracy por tipo de tarea (triage / NER / SBAR)
for cat in ["triage", "ner", "sbar"]:
    cols = [
        f"{n}_acc" for n in OP_NAMES if n.startswith(cat) and f"{n}_acc" in ok.columns
    ]
    if cols:
        ok[f"{cat}_acc"] = ok[cols].mean(axis=1)

cat_cols = [c for c in ["triage_acc", "ner_acc", "sbar_acc"] if c in ok.columns]
ok[["model_short", *cat_cols, "mean_acc"]].sort_values("mean_acc", ascending=False)

## Selección del mejor modelo

Score compuesto normalizado `[0, 1]`:

- **calidad** (`mean_acc`) — mayor es mejor
- **velocidad** (`mean_op_ms`) — menor es mejor
- **memoria** (`mem_after_load_mb`) — menor es mejor

Ajusta los pesos según prioridad del despliegue en la Beelink.

In [ ]:
# 7) Score compuesto y elección del mejor modelo
W_ACC, W_SPEED, W_MEM = 0.5, 0.3, 0.2  # pesos (suman 1.0)


def norm_low(s):  # menor = mejor -> mapea a [0,1] con 1 = mejor
    rng = s.max() - s.min()
    return (s.max() - s) / rng if rng else pd.Series(1.0, index=s.index)


ok["score_acc"] = ok["mean_acc"].fillna(0)  # ya está en [0,1]
ok["score_speed"] = norm_low(ok["mean_op_ms"])
ok["score_mem"] = norm_low(ok["mem_after_load_mb"])
ok["score"] = (
    W_ACC * ok["score_acc"] + W_SPEED * ok["score_speed"] + W_MEM * ok["score_mem"]
)

ranking = ok.sort_values("score", ascending=False).reset_index(drop=True)
best = ranking.iloc[0]

print(f"🏆 Mejor modelo: {best['model_short']}  (score = {best['score']:.3f})")
print(f"   accuracy media : {best['mean_acc']:.3f}")
print(f"   latencia/op    : {best['mean_op_ms']:.0f} ms")
print(f"   RAM tras carga : {best['mem_after_load_mb']:.0f} MB")
print(f"   tiempo de carga: {best['load_time_ms'] / 1000:.1f} s")

ranking[
    [
        "model_short",
        "score",
        "mean_acc",
        "mean_op_ms",
        "mem_after_load_mb",
        "load_time_ms",
    ]
]

## Imágenes comparativas

Se guardan en `./figures/`.

In [ ]:
# 8) Carpeta de figuras + helper
FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print("guardado:", path)

In [ ]:
# 9) Calidad (accuracy media por modelo)
r = ranking.sort_values("mean_acc")
fig, ax = plt.subplots()
ax.barh(r["model_short"], r["mean_acc"], color="#2a9d8f")
ax.set_xlabel("Accuracy media (0–1)")
ax.set_xlim(0, 1)
ax.set_title("Calidad por modelo (triage + NER + SBAR)")
for i, v in enumerate(r["mean_acc"]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "01_accuracy.png")
plt.show()

In [ ]:
# 10) Rendimiento de inferencia (latencia media por operación, CPU)
r = ranking.sort_values("mean_op_ms", ascending=False)
fig, ax = plt.subplots()
ax.barh(r["model_short"], r["mean_op_ms"], color="#e76f51")
ax.set_xlabel("Latencia media por operación (ms) — menor es mejor")
ax.set_title("Rendimiento de inferencia (100% CPU)")
for i, v in enumerate(r["mean_op_ms"]):
    ax.text(v, i, f" {v:.0f}", va="center")
save(fig, "02_performance.png")
plt.show()

In [ ]:
# 11) Memoria: RAM tras carga vs tamaño de archivo, con el presupuesto marcado
r = ranking.sort_values("mem_after_load_mb")
x = np.arange(len(r))
w = 0.4
fig, ax = plt.subplots()
ax.bar(x - w / 2, r["mem_after_load_mb"], w, label="RAM tras carga", color="#264653")
ax.bar(x + w / 2, r["file_size_mb"], w, label="Tamaño de archivo", color="#8ecae6")
ax.axhline(
    lp.MAX_RAM_GB * 1024,
    color="red",
    ls="--",
    label=f"Presupuesto {lp.MAX_RAM_GB:.0f} GB",
)
ax.set_xticks(x)
ax.set_xticklabels(r["model_short"], rotation=45, ha="right")
ax.set_ylabel("MB")
ax.set_title("Memoria por modelo")
ax.legend()
save(fig, "03_memory.png")
plt.show()

In [ ]:
# 12) Uso de CPU (inferencia 100% CPU): tiempo de carga y cómputo total
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rl = ranking.sort_values("load_time_ms", ascending=False)
axes[0].barh(rl["model_short"], rl["load_time_ms"] / 1000, color="#f4a261")
axes[0].set_xlabel("Tiempo de carga (s)")
axes[0].set_title("Carga del modelo en CPU")

rc = ranking.sort_values("total_ops_ms", ascending=False)
axes[1].barh(rc["model_short"], rc["total_ops_ms"] / 1000, color="#e9c46a")
axes[1].set_xlabel("Tiempo total de cómputo (s)")
axes[1].set_title("Trabajo de CPU — todas las operaciones")

fig.suptitle("Uso de CPU (sin GPU)", fontsize=13)
save(fig, "04_cpu.png")
plt.show()

In [ ]:
# 13) Trade-off calidad vs velocidad (tamaño de burbuja = RAM, color = score)
fig, ax = plt.subplots()
sizes = (ranking["mem_after_load_mb"] / ranking["mem_after_load_mb"].max()) * 900 + 80
sc = ax.scatter(
    ranking["mean_op_ms"],
    ranking["mean_acc"],
    s=sizes,
    c=ranking["score"],
    cmap="viridis",
    alpha=0.85,
    edgecolors="k",
)
for _, row in ranking.iterrows():
    ax.annotate(
        row["model_short"],
        (row["mean_op_ms"], row["mean_acc"]),
        fontsize=8,
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_xlabel("Latencia media por op (ms) — menor mejor")
ax.set_ylabel("Accuracy media — mayor mejor")
ax.set_title("Calidad vs velocidad (burbuja = RAM)")
plt.colorbar(sc, label="score compuesto")
save(fig, "05_tradeoff.png")
plt.show()

In [ ]:
# 14) Heatmap de accuracy por operación
heat = ranking.set_index("model_short")[acc_cols]
heat.columns = [c.replace("_acc", "") for c in heat.columns]
fig, ax = plt.subplots(figsize=(max(8, len(acc_cols) * 0.9), 0.6 * len(heat) + 2))
im = ax.imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="accuracy")
ax.set_title("Accuracy por operación")
save(fig, "06_accuracy_heatmap.png")
plt.show()

In [ ]:
# 15) Ranking final (score compuesto) — el mejor resaltado
fig, ax = plt.subplots()
colors = ["#2a9d8f" if i == 0 else "#90be6d" for i in range(len(ranking))]
ax.barh(ranking["model_short"][::-1], ranking["score"][::-1], color=colors[::-1])
ax.set_xlabel("Score compuesto (mayor = mejor)")
ax.set_title(f"Ranking final  —  🏆 {best['model_short']}")
for i, v in enumerate(ranking["score"][::-1]):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center")
save(fig, "07_ranking.png")
plt.show()

In [ ]:
# 16) Resumen y recomendación
print("=" * 56)
print("RESUMEN DEL ANÁLISIS")
print("=" * 56)
print("Origen            : lp.run_benchmark() (DataFrame en memoria, sin CSV)")
print(f"Modelos evaluados : {len(ok)} (de {len(df)} en el DataFrame)")
print(f"Pesos del score   : accuracy={W_ACC} · velocidad={W_SPEED} · memoria={W_MEM}")
print("-" * 56)
print(f"RECOMENDADO       : {best['model_short']}")
print(f"  score compuesto : {best['score']:.3f}")
print(f"  accuracy media  : {best['mean_acc']:.3f}")
print(f"  latencia / op   : {best['mean_op_ms']:.0f} ms")
print(f"  RAM tras carga  : {best['mem_after_load_mb']:.0f} MB")
print(f"  tiempo de carga : {best['load_time_ms'] / 1000:.1f} s")
print("-" * 56)
print(f"Figuras guardadas en: {FIG_DIR}")